In [1]:
import optuna
from tensorflow import keras
from keras import layers
from sklearn.metrics import roc_auc_score
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from joblib import dump

/home/prateek/Prateek/LaunchPad/week6/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-01-30 11:14:39.918046: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-30 11:14:39.918346: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-30 11:14:39.960257: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebu

In [2]:
df = pd.read_csv('/home/prateek/Prateek/LaunchPad/week6/Day3/src/data/processed/final.csv')

X = df.drop('Survived', axis=1).values
y = df['Survived'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)
np.save('/home/prateek/Prateek/LaunchPad/week6/Day4/src/data/processed/x_train.npy', X_train)
np.save('/home/prateek/Prateek/LaunchPad/week6/Day4/src/data/processed/x_test.npy', X_test)
np.save('/home/prateek/Prateek/LaunchPad/week6/Day4/src/data/processed/y_train.npy', y_train)
np.save('/home/prateek/Prateek/LaunchPad/week6/Day4/src/data/processed/y_test.npy', y_test)


(712, 5) (179, 5)


In [3]:
def build_model(trial, input_dim):
    n_layers = trial.suggest_int("n_layers", 1, 3)
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))

    for i in range(n_layers):
        units = trial.suggest_int(f"units_{i}", 16, 128, step=16)
        dropout = trial.suggest_float(f"dropout_{i}", 0.1, 0.5)
        model.add(layers.Dense(units, activation="relu"))
        model.add(layers.Dropout(dropout))

    model.add(layers.Dense(1, activation="sigmoid"))

    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model


In [4]:
def make_objective(X_train, y_train, X_test, y_test):
    def objective(trial):
        model = build_model(trial, X_train.shape[1])

        batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])

        early_stopping = keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True
        )

        model.fit(
            X_train, y_train,
            validation_split=0.2,
            epochs=100,
            batch_size=batch_size,
            callbacks=[early_stopping],
            verbose=0
        )

        y_pred_proba = model.predict(X_test).flatten()
        auc = roc_auc_score(y_test, y_pred_proba)

        return auc

    return objective


In [5]:
objective_fn = make_objective(X_train, y_train, X_test, y_test)
study = optuna.create_study(direction="maximize")
study.optimize(objective_fn, n_trials=50)


[I 2026-01-30 11:14:42,825] A new study created in memory with name: no-name-8f450839-449b-42fd-8995-478260bf0fc7


2026-01-30 11:14:42.833268: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 


[I 2026-01-30 11:14:50,730] Trial 0 finished with value: 0.8480237154150198 and parameters: {'n_layers': 2, 'units_0': 80, 'dropout_0': 0.3252980433759356, 'units_1': 64, 'dropout_1': 0.12946618492139272, 'lr': 0.00013691273626576439, 'batch_size': 64}. Best is trial 0 with value: 0.8480237154150198.


6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 


...

[I 2026-01-30 11:17:52,924] Trial 47 finished with value: 0.8345849802371541 and parameters: {'n_layers': 2, 'units_0': 96, 'dropout_0': 0.20941311796342915, 'units_1': 128, 'dropout_1': 0.25634796857703934, 'lr': 0.008184365997320586, 'batch_size': 16}. Best is trial 42 with value: 0.8528985507246377.


6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 


[I 2026-01-30 11:17:55,662] Trial 48 finished with value: 0.8468379446640316 and parameters: {'n_layers': 2, 'units_0': 128, 'dropout_0': 0.29753688064415773, 'units_1': 80, 'dropout_1': 0.2046593036499384, 'lr': 0.001213101240428233, 'batch_size': 64}. Best is trial 42 with value: 0.8528985507246377.


6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 


[I 2026-01-30 11:17:57,618] Trial 49 finished with value: 0.8447299077733861 and parameters: {'n_layers': 2, 'units_0': 112, 'dropout_0': 0.3498431779203356, 'units_1': 48, 'dropout_1': 0.10124532854390644, 'lr': 0.0043647924697348866, 'batch_size': 32}. Best is trial 42 with value: 0.8528985507246377.


In [6]:
print("Best AUC:", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"{k}: {v}")


Best AUC: 0.8528985507246377
Best params:
n_layers: 2
units_0: 80
dropout_0: 0.3038974062289378
units_1: 80
dropout_1: 0.12780323185196837
lr: 0.00019932691508874073
batch_size: 64


In [7]:
from keras.callbacks import EarlyStopping
best_params = study.best_params

def build_best_model(input_dim):
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))

    for i in range(best_params["n_layers"]):
        model.add(layers.Dense(
            best_params[f"units_{i}"],
            activation="relu"
        ))
        model.add(layers.Dropout(best_params[f"dropout_{i}"]))

    model.add(layers.Dense(1, activation="sigmoid"))

    model.compile(
        optimizer=keras.optimizers.Adam(best_params["lr"]),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

final_model = build_best_model(X_train.shape[1])

final_model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=best_params["batch_size"],
    validation_split=0.2,
    callbacks=[EarlyStopping(patience=10, restore_best_weights=True)],
    verbose=1
)


Epoch 1/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.5852 - loss: 0.6684 - val_accuracy: 0.5944 - val_loss: 0.6543
Epoch 2/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6344 - loss: 0.6413 - val_accuracy: 0.5944 - val_loss: 0.6415
Epoch 3/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6309 - loss: 0.6333 - val_accuracy: 0.5944 - val_loss: 0.6332
Epoch 4/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6292 - loss: 0.6291 - val_accuracy: 0.5944 - val_loss: 0.6274
Epoch 5/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6397 - loss: 0.6197 - val_accuracy: 0.5944 - val_loss: 0.6229
Epoch 6/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6380 - loss: 0.6053 - val_accuracy: 0.5944 - val_loss: 0.6191
Epoch 7/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6362 - loss: 0.6115 - val_accuracy: 0.5944 - val_loss: 0.6155
Epoch 8/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6450 - loss: 0.6039 - val_accuracy: 0.5944 - val_loss: 0.6118

In [8]:
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)
import json
import numpy as np


In [9]:
# Predictions
y_train_proba = final_model.predict(X_train).flatten()
y_test_proba = final_model.predict(X_test).flatten()

y_train_pred = (y_train_proba > 0.5).astype(int)
y_test_pred = (y_test_proba > 0.5).astype(int)

# Metrics
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

train_auc = roc_auc_score(y_train, y_train_proba)
test_auc = roc_auc_score(y_test, y_test_proba)

cm = confusion_matrix(y_test, y_test_pred)


23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 


In [10]:
results = {
    "model": "NeuralNetwork",
    "dataset": "Titanic",
    "best_params": best_params,

    "metrics": {
        "train_accuracy": float(train_accuracy),
        "test_accuracy": float(test_accuracy),
        "train_roc_auc": float(train_auc),
        "test_roc_auc": float(test_auc)
    },

    "confusion_matrix": {
        "tn": int(cm[0, 0]),
        "fp": int(cm[0, 1]),
        "fn": int(cm[1, 0]),
        "tp": int(cm[1, 1])
    }
}


In [11]:
results_path = "/home/prateek/Prateek/LaunchPad/week6/Day4/src/Tunning/result.json"

with open(results_path, "w") as f:
    json.dump(results, f, indent=4)

print(f"Results saved to {results_path}")


Results saved to /home/prateek/Prateek/LaunchPad/week6/Day4/src/Tunning/result.json


In [12]:
import os

models_dir = "models"
os.makedirs(models_dir, exist_ok=True)

h5_path = os.path.join(models_dir, "final_model.h5")
pkl_path = os.path.join(models_dir, "final_model.pkl")
final_model.save(h5_path)
dump(final_model, pkl_path)

print(f"Model saved to {h5_path}")


Model saved to models/final_model.h5
